### Setup

In [ ]:
import random

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from dogbreeds.paths import DATA_DIR, OUTPUTS_DIR, PROJECT_ROOT, RAW_DIR

# Figures saved here are committed for later use in README and data card.
FIGURES_DIR = PROJECT_ROOT / "docs" / "figures"
FIGURES_DIR.mkdir(exist_ok=True, parents=True)

# Train / val /test colors
SPLIT_COLORS = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}
GRID_COLOR = "#e5e5e5"
MUTED_TEXT = "#52514e"

splits = pd.read_csv(DATA_DIR / "splits.csv")
inventory = pd.read_csv(OUTPUTS_DIR / "inventory.csv")
inventory = inventory[inventory["status"] == "ok"]
print(f"{len(splits)} images in splits.csv, {len(inventory)} usable images in the inventory.")

### Class Balance

In [ ]:
# Images per breed, stacked by split and sorted by size.
counts = splits.groupby(["breed", "split"]).size().unstack()[["train", "val", "test"]]
totals = counts.sum(axis=1).sort_values()
counts = counts.loc[totals.index]

fig, ax = plt.subplots(figsize=(8, 5))
left = pd.Series(0, index=counts.index)
for split in ["train", "val", "test"]:
    ax.barh(
        counts.index,
        counts[split],
        left=left,
        color=SPLIT_COLORS[split],
        label=split,
        edgecolor="white",
        linewidth=1,
    )
    left = left + counts[split]

# Write each breed's total at the end of its bar.
for row, total in enumerate(totals):
    ax.text(total + 3, row, str(total), va="center", fontsize=9, color=MUTED_TEXT)

ax.set_title("Images per breed and split")
ax.set_xlabel("Number of images")
ax.set_ylabel("Breed")
ax.legend(loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color=GRID_COLOR)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "class_balance.png", dpi=150)
plt.show()
print(f"Largest / smallest breed: {totals.max() / totals.min():.2f}x")

### Image Sizes

In [ ]:
# The model sees 224x224 images.
shortest_side = inventory[["width", "height"]].min(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(shortest_side, bins=40, color=SPLIT_COLORS["train"], edgecolor="white")
ax.axvline(224, color=MUTED_TEXT, linestyle="--", linewidth=1.5)
ax.text(232, ax.get_ylim()[1] * 0.9, "224 px", color=MUTED_TEXT)
ax.set_title("Shortest side of each image")
ax.set_xlabel("Pixels")
ax.set_ylabel("Images")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "image_sizes.png", dpi=150)
plt.show()
print(f"{(shortest_side < 224).sum()} of {len(shortest_side)} images are smaller than 224 px")

### Helper Functions

In [ ]:
def show_grid(images, titles, columns, figure_title, save_name):
    """Show images ('breed/file.jpg') in a grid with a small ttitle under each."""
    rows = (len(images) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(columns * 2, rows * 2.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for i, image in enumerate(images):
        ax = axes.flat[i]
        with Image.open(RAW_DIR / image) as img:
            ax.imshow(img.convert("RGB"))
        ax.set_title(titles[i], fontsize=8)
    fig.suptitle(figure_title)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / save_name, dpi=100, pil_kwargs={"quality": 80})
    plt.show()


def sample_train_images(breed, count, rng):
    """Pick random training images of one breed (never val/test)."""
    is_breed = splits["breed"] == breed
    is_train = splits["split"] == "train"
    images = splits[is_breed & is_train]["image"].tolist()
    return rng.sample(images, count)

### Sample Images Per Breed

In [ ]:
rng = random.Random(42)
images = []
titles = []
for breed in sorted(splits["breed"].unique()):
    for image in sample_train_images(breed, 6, rng):
        images.append(image)
        titles.append(breed)
show_grid(images, titles, 6, "Six random training images per breed", "samples_per_breed.jpg")

### Look Alike Groups Side by Side

In [ ]:
LOOK_ALIKE_GROUPS = [
    ["maltese", "shih_tzu", "yorkshire_terrier"],
    ["basset", "bloodhound"],
    ["siberian_husky", "german_shepherd"],
]
rng = random.Random(43)
for number, group in enumerate(LOOK_ALIKE_GROUPS, start=1):
    images = []
    titles = []
    for breed in group:
        for image in sample_train_images(breed, 4, rng):
            images.append(image)
            titles.append(breed)
    show_grid(images, titles, 4, " vs ".join(group), f"look_alikes_{number}.jpg")

Basset vs. Bloodhound and Maltese vs. Shih-Tzu seem to be the hardest paris as the dogs are very similar to one another. Basset and Bloodhound  only really differ by their colors and Maltese and Shih-Tzu have slightly differing hair patterns. The background of these look-alike images and the earlier sample images are similar. Most images just feature the dog centered with a plain background, but some have a background of a grassy field, rocky terrain, a beach, people, etc. Overall the differences in background are pretty evently distributed across the breeds, so the model most likely won't pick up on any background similariites. 